In [6]:
# ==========================================
# 1. 環境初始化與套件安裝
# ==========================================
!pip install pymupdf pandas openpyxl

import fitz
import pandas as pd
import re
import math
import os
from google.colab import files  # Colab 專用上傳下載庫
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter
from openpyxl.cell.rich_text import TextBlock, CellRichText
from openpyxl.cell.text import InlineFont
from openpyxl.worksheet.datavalidation import DataValidation

# --- 檔案名稱設定區 (請確保上傳的檔名一致) ---
EXCEL_FILE = "BOM.xlsx"
PDF_FILE = "SCH.pdf"
OUTPUT_EXCEL = "SCH & BOM 比對結果.xlsx"
COLUMN_NAME = "Designator"

# ==========================================
# 2. 輔助函式定義
# ==========================================

def apply_mixed_font(cell, text, size=11, bold=False, color="4F4F4F"):
    """莫蘭迪風格：微軟正黑體 + Segoe UI 混排"""
    font_zh = InlineFont(rFont='微軟正黑體', sz=size, b=bold, color=color)
    font_en = InlineFont(rFont='Segoe UI', sz=size, b=bold, color=color)
    rt = CellRichText()
    parts = re.split(r'([\u4e00-\u9fff]+)', text)
    for part in parts:
        if not part: continue
        if re.search(r'[\u4e00-\u9fff]', part):
            rt.append(TextBlock(font_zh, part))
        else:
            rt.append(TextBlock(font_en, part))
    cell.value = rt

def get_sort_key(refdes):
    """零件類別排序邏輯"""
    ref_str = str(refdes).strip()
    if not ref_str or ref_str.lower() == 'nan': return (9999, 0, ref_str)
    match = re.match(r"([A-Za-z]+)([0-9]+)?", ref_str)
    SORT_ORDER = ["PCB","inlet", "F", "CX", "CK","CR", "L", "LF", "T", "RA", "RB", "RC", "RD", "RE", "R", "RX", "M", "TH", "C", "CY", "B", "BD", "ACD", "ZD", "D", "Q", "PC", "U"]
    order_lower = [s.lower() for s in SORT_ORDER]
    if not match: return (-2, 0, ref_str)
    prefix, num_str = match.groups()
    prefix_lower = prefix.lower()
    num = int(num_str) if num_str else 0
    category_weight = order_lower.index(prefix_lower) if prefix_lower in order_lower else 999
    return (category_weight, num)

def get_distance_downward(ref_rect, other_rect):
    """計算參考點到其他文字的權重距離"""
    ref_cy, ref_cx = (ref_rect.y0 + ref_rect.y1) / 2, (ref_rect.x0 + ref_rect.x1) / 2
    oth_cy, oth_cx = (other_rect.y0 + other_rect.y1) / 2, (other_rect.x0 + other_rect.x1) / 2
    if oth_cy < ref_cy: return float('inf')
    return math.sqrt(((oth_cx - ref_cx) * 1.5)**2 + (oth_cy - ref_cy)**2)

# ==========================================
# 3. 主程序執行邏輯
# ==========================================

def start_process():
    # --- 步驟 A: 檔案上傳 ---
    print("🚀 STEP 1: 請上傳您的 BOM.xlsx 與 SCH.pdf 檔案")
    uploaded = files.upload()

    if EXCEL_FILE not in uploaded or PDF_FILE not in uploaded:
        print(f"❌ 錯誤：請確認上傳的檔案名稱是否精確為 '{EXCEL_FILE}' 與 '{PDF_FILE}'")
        return

    print(f"🔍 STEP 2: 開始分析 PDF 檔案...")

    try:
        # 1. 讀取 BOM 資料
        df_bom = pd.read_excel(EXCEL_FILE)
        new_headers = [str(df_bom.iloc[0, i]) for i in range(4)]
        df_data = df_bom[df_bom[COLUMN_NAME].notna()].copy().drop(df_bom.index[0])
        designators = df_data[COLUMN_NAME].astype(str).str.strip()
        maps = [dict(zip(designators, df_data.iloc[:, i])) for i in range(4)]
        bom_refdes = [ref for ref in designators.unique() if ref.lower() not in ['location', 'nan', '']]

        results_map = {ref: {"元件值": "未找到", "元件封裝": "未找到", "是否在PDF出現": "❌ 否",
                             "orig": [m.get(ref, "") for m in maps]} for ref in bom_refdes}

        # 2. PDF 處理
        doc = fitz.open(PDF_FILE)
        page_images = []
        PKG_PATTERNS = r"0201|0402|0603|0805|1206|SOT|QFN|SOD|DO-|RES-|CAP-|TQFP|BGA|LED"
        REF_PATTERN = r"^[A-Za-z]+[0-9]+(-[A-Z0-9]+)?$"

        for i, page in enumerate(doc):
            all_words = [{"text": s["text"].strip(), "rect": fitz.Rect(s["bbox"])}
                         for b in page.get_text("dict")["blocks"] if "lines" in b
                         for l in b["lines"] if l["dir"] == (1.0, 0.0)
                         for s in l["spans"]]

            merged = []
            skip = set()
            for k, w in enumerate(all_words):
                if k in skip: continue
                txt, rct = w["text"], w["rect"]
                for j, nw in enumerate(all_words):
                    if k != j and j not in skip and abs(rct.y0 - nw["rect"].y0) < 2 and 0 < (nw["rect"].x0 - rct.x1) < 6:
                        txt += nw["text"]; rct = rct | nw["rect"]; skip.add(j)
                merged.append({"text": txt, "rect": rct})

            for ref in bom_refdes:
                for item in merged:
                    if (item["text"] == ref) or (re.fullmatch(re.escape(ref) + r"-[A-Z0-9]+", item["text"])):
                        results_map[ref]["是否在PDF出現"] = "✅ 是"
                        cands = []
                        search_area = item["rect"] + (-80, -80, 80, 80)
                        for o in merged:
                            if o["text"] != item["text"] and not re.match(REF_PATTERN, o["text"]) and search_area.intersects(o["rect"]):
                                d = get_distance_downward(item["rect"], o["rect"])
                                if d != float('inf'): cands.append((d, o["text"]))

                        cands.sort(key=lambda x: x[0])
                        if len(cands) > 0:
                            if len(cands) >= 2:
                                is_first_pkg = re.search(PKG_PATTERNS, cands[0][1], re.I)
                                if is_first_pkg:
                                    results_map[ref]["元件值"], results_map[ref]["元件封裝"] = cands[1][1], cands[0][1]
                                else:
                                    results_map[ref]["元件值"], results_map[ref]["元件封裝"] = cands[0][1], cands[1][1]
                            else:
                                results_map[ref]["元件值"] = cands[0][1]

                        annot = page.add_highlight_annot(item["rect"])
                        if annot:
                            annot.set_colors(stroke=(1, 1, 0)) # 亮黃色
                            annot.update()

            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img_path = f"temp_sch_page_{i}.png"
            pix.save(img_path)
            page_images.append(img_path)

        # 3. 整理與排序
        final_list = [{"元件代號": k, "元件值": v["元件值"], "元件封裝": v["元件封裝"], "是否在PDF出現": v["是否在PDF出現"],
                       new_headers[0]: v["orig"][0], new_headers[1]: v["orig"][1],
                       new_headers[2]: v["orig"][2], new_headers[3]: v["orig"][3]} for k, v in results_map.items()]
        final_df = pd.DataFrame(final_list)
        final_df['sort_key'] = final_df['元件代號'].apply(get_sort_key)
        final_df = final_df.sort_values(by='sort_key').drop(columns=['sort_key'])

        # 4. Excel 輸出、格式化、釘選、不換行
        print(f"📊 STEP 3: 正在產出報表並設定格式...")
        with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
            final_df.to_excel(writer, index=False, sheet_name='CheckList', startrow=2)
            ws = writer.sheets['CheckList']
            max_row = ws.max_row
            ws.freeze_panes = 'A4' # 釘選前三列

            M_BLUE_DARK, M_BLUE_LIGHT, M_PINK, M_GREEN, M_BEIGE, M_TEXT, M_BORDER = "8294A5", "E1E7ED", "E8D7D7", "CFD8D0", "F2F1EF", "4F4F4F", "D1D1D1"

            # 說明欄
            ws.merge_cells('A1:H2')
            header_text = ("【SCH & BOM 比對報告】 A~C 欄：電路圖自動擷取資訊 │ E~H 欄：原始 BOM 對應資訊。\n"
                           " ⚠️ D欄標註「❌ 否」代表未偵測到。第三列已開啟下拉篩選，D欄可手動調整狀態。")
            apply_mixed_font(ws['A1'], header_text, size=11, bold=True, color="FFFFFF")
            ws['A1'].fill = PatternFill(start_color=M_BLUE_DARK, end_color=M_BLUE_DARK, fill_type="solid")
            ws['A1'].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            ws.row_dimensions[1].height = 25
            ws.row_dimensions[2].height = 25

            dv = DataValidation(type="list", formula1='"✅ 是,❌ 否"', allow_blank=True)
            ws.add_data_validation(dv)
            dv.add(f'D4:D{max_row}')
            ws.auto_filter.ref = f"A3:{get_column_letter(ws.max_column)}{max_row}"

            column_widths = {}
            for row in ws.iter_rows(min_row=3, max_row=max_row):
                r_idx = row[0].row
                ws.row_dimensions[r_idx].height = 22
                for cell in row:
                    cell.border = Border(left=Side(style='thin', color=M_BORDER), right=Side(style='thin', color=M_BORDER),
                                         top=Side(style='thin', color=M_BORDER), bottom=Side(style='thin', color=M_BORDER))
                    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=False) # 禁止換行

                    val = str(cell.value) if cell.value is not None else ""
                    current_width = sum(2.3 if re.search(r'[\u4e00-\u9fff]', c) else 1.3 for c in val)
                    if r_idx == 3: current_width += 8
                    column_widths[cell.column] = max(column_widths.get(cell.column, 0), current_width)

                    if r_idx == 3:
                        cell.fill = PatternFill(start_color=M_BLUE_DARK, end_color=M_BLUE_DARK, fill_type="solid")
                        cell.font = Font(name='Segoe UI', bold=True, color="FFFFFF", size=12)
                    else:
                        is_missing = ("❌" in str(row[3].value))
                        if cell.column <= 3: cell.fill = PatternFill(start_color=M_BLUE_LIGHT, end_color=M_BLUE_LIGHT, fill_type="solid")
                        elif cell.column == 4: cell.fill = PatternFill(start_color=M_PINK, end_color=M_PINK, fill_type="solid")
                        else: cell.fill = PatternFill(start_color=M_GREEN, end_color=M_GREEN, fill_type="solid")

                        f_color = "B55D5D" if is_missing else M_TEXT
                        f_name = '微軟正黑體' if re.search(r'[\u4e00-\u9fff]', val) else 'Segoe UI'
                        cell.font = Font(name=f_name, size=11, color=f_color)

            for col_idx, final_w in column_widths.items():
                ws.column_dimensions[get_column_letter(col_idx)].width = final_w

            # 視覺化分頁
            ws2 = writer.book.create_sheet(title="ResultVisual")
            ws2.merge_cells('A1:M3')
            visual_header = "【電路圖目視檢驗表單】\n 🔹 亮黃色高亮標記：已存在於 BOM 清單中。 🔹 無標記：圖中存在但 BOM 遺漏之元件。"
            apply_mixed_font(ws2['A1'], visual_header, size=11, bold=True)
            ws2['A1'].fill = PatternFill(start_color=M_BEIGE, end_color=M_BEIGE, fill_type="solid")
            ws2['A1'].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            ws2.row_dimensions[1].height = 45
            ws2.freeze_panes = 'A4'

            current_row = 5
            for img_path in page_images:
                img = Image(img_path)
                img.width, img.height = 1100, img.height * (1100 / img.width)
                ws2.add_image(img, f'B{current_row}')
                row_span = int(img.height / 15)
                current_row += row_span + 2

        doc.close()
        for p in page_images:
            if os.path.exists(p): os.remove(p)

        print(f"✨ 任務完成！報表：{OUTPUT_EXCEL}")

        # 5. 自動下載結果
        files.download(OUTPUT_EXCEL)

    except Exception as e:
        print(f"❌ 執行中發生錯誤: {e}")

if __name__ == "__main__":
    start_process()

🚀 STEP 1: 請上傳您的 BOM.xlsx 與 SCH.pdf 檔案


KeyboardInterrupt: 